# 03 — Generative UI

**Goal:** let an agent decide not only *what* the answer is, but *how it should be shown*.

Most agents return text. Sometimes text is not the best shape for the answer. A list of numbers wants to be a **chart**. A grid of records wants to be a **table**. A multi-step process wants to be a **plan widget** that updates in real time.

In this notebook you will:

1. See the 17 built-in UI component types orqest ships with.
2. Build a small agent that produces structured data (a labelled dataset).
3. Wrap that data as a typed `ChartComponent` and send it onto the event bus.
4. Watch a live `ExecutionPlan` widget update as task statuses change.
5. Hook everything up to an SSE stream — exactly what a real web frontend would consume.

The big idea: **the agent makes the data; the UI layer is a separate concern**. The agent never imports a UI library.

**Before you start:** you need a `.env` file at the repo root with `LLM_API_KEY` and `LLM_MODEL`.


## 1. Load the config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The Workbench carries the component registry

A `Workbench` is a bag of runtime tools. It holds five things:

- a **memory** store,
- a **tracer** for spans and timing,
- an **event bus** for pub/sub,
- a **recent events** buffer (so consumers can replay missed events),
- a **component registry** that knows how to validate UI components.

When you create a new `Workbench`, the registry is pre-loaded with 17 first-party components. They live across three layers: simple primitives like `text` and `button`, declarative grammars like `vega_chart` and `mermaid`, and an escape hatch called `sandboxed_html`.

Let's see them all.


In [2]:
import tempfile, os
from orqest.workbench import Workbench
from orqest.memory import LocalMemoryStore

wb = Workbench(memory=LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "ui.db")))

types = wb.ui_registry.list_types()
print(f"{len(types)} first-party component types:")
print("  " + ", ".join(types))

17 first-party component types:
  badge, button, chart, form, image, input, json_viewer, latex, layout, markdown, mermaid, plan, sandboxed_html, table, takeover_dialog, text, vega_chart


## 3. An agent that produces structured data

The agent's job is the **content**. The UI layer is the **surface**. These are two separate responsibilities, and keeping them apart is what makes the system flexible.

For the demo we use a plain `BaseAgent` that takes a topic and returns a small labelled dataset — exactly the shape a bar chart expects. Three things to notice:

- The output type `Dataset` is a normal Pydantic model.
- The agent system prompt is short and specific: it asks for 4–6 data points.
- There is **nothing** UI-related in the agent. No chart, no colour, no styling.


In [ ]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


class DataPoint(BaseModel):
    label: str
    value: float


class Dataset(BaseModel):
    title: str = Field(description="A short chart title.")
    series_name: str = Field(description="What the values measure.")
    points: list[DataPoint] = Field(description="4-6 labelled data points.")


class DataAgent(BaseAgent[GlobalState, Dataset]):
    def __init__(self, model, api_key=None):
        super().__init__(
            agent_name="data_agent",
            system_prompt=(
                "You turn a topic into a small, plausible labelled dataset "
                "suitable for a bar chart. 4-6 points. Values are illustrative."
            ),
            output_type=Dataset,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> Dataset:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


data_agent = DataAgent(model=config.llm_model, api_key=config.llm_api_key)
print("Data agent ready.")

Now run the agent on a real topic. It returns a `Dataset` — pure data, ready to be drawn.

In [3]:
state = GlobalState()
state.add_message("user", "Approximate share of developers using each major code editor.")
dataset = await data_agent.run(state)

print(dataset.title)
for p in dataset.points:
    print(f"  {p.label:20s} {p.value}")

Approximate Share of Developers by Code Editor
  VS Code              74.0
  Visual Studio        17.0
  IntelliJ IDEA        15.0
  Vim                  11.0
  Sublime Text         6.0
  Notepad++            5.0


## 4. Wrap the data as a typed UI component

This is the key step. The `Dataset` is just data — we now turn it into a `ChartComponent`. A `ChartComponent` is a typed envelope that says: "this is meant to be rendered as a chart, here is the data, here are the chart-specific options."

The component is then handed to a `UIEmitter`, which publishes it onto the workbench's event bus as a `ui.chart.init` event. Anything that subscribes to that bus will see it: a frontend, an SSE stream, a logger, a test.

We subscribe to all events first so you can see the exact wire format.


In [ ]:
from orqest.ui import UIEmitter, ChartComponent, ChartComponentData, ChartSeries
from orqest.observability import AgentEvent

# Collect every event that crosses the bus so we can inspect them.
seen: list[AgentEvent] = []
wb.event_bus.subscribe_all(seen.append)

print("Listening on the event bus.")

Build the chart spec from the dataset, then emit it. Notice the structure:

- `component_id` — a stable identifier (the frontend uses this to know which chart to update later).
- `data.chart_kind` — the chart sub-type (bar, line, pie, etc.).
- `data.series` — the actual values, in a generic "list of points" shape.


In [4]:
chart = ChartComponent(
    component_id="editor-share",
    data=ChartComponentData(
        chart_kind="bar",
        title=dataset.title,
        series=[ChartSeries(
            name=dataset.series_name,
            points=[{"x": p.label, "y": p.value} for p in dataset.points],
        )],
    ),
)

emitter = UIEmitter(wb.event_bus, agent_name="data_agent")
await emitter.init(chart)

# Inspect what landed on the bus.
evt = seen[-1]
print(f"event_type:   {evt.event_type}")
print(f"payload keys: {sorted(evt.data.keys())}")
print(f"chart_kind:   {evt.data['data']['chart_kind']}")
print(f"points:       {len(evt.data['data']['series'][0]['points'])}")

event_type: ui.chart.init
payload keys: ['component_id', 'component_type', 'created_at', 'data', 'metadata']
chart_kind:  bar
points:      6


## 5. Streaming updates — a live plan widget

Some components are not static. An `ExecutionPlan` is a task list, and tasks change status over time: `pending` → `in-progress` → `completed`. We want the UI to update as that happens, not just once at the end.

The way this works:

- `enable_ui_events()` turns on dual emission. The plan still fires its legacy `plan.*` events, **and** also fires typed `ui.plan.*` events. Existing consumers keep working; new ones get the richer typed stream.
- `emit_init` sends the initial state as a `ui.plan.init` event (the full plan).
- Every `set_task_status` then sends a small `ui.plan.delta` event with only the part that changed (a JSON path like `tasks.0.status`).

Small deltas mean a frontend can update one row without rebuilding the whole widget.


In [ ]:
from orqest.plan import ExecutionPlan, PlanTask

plan = ExecutionPlan(tasks=[
    PlanTask(id="t1", title="Generate the dataset"),
    PlanTask(id="t2", title="Render the chart"),
]).enable_ui_events(component_id="render-plan")

print(f"Plan has {len(plan.tasks)} tasks. UI events enabled.")

Now send the initial state, then mark one task complete and the other in progress. Watch the events fire.

In [5]:
await plan.emit_init(wb.event_bus, agent_name="ui_agent")
await plan.set_task_status("t1", "completed", bus=wb.event_bus, agent_name="ui_agent")
await plan.set_task_status("t2", "in-progress", bus=wb.event_bus, agent_name="ui_agent")

# Both channels fire: legacy `plan.*` and typed `ui.plan.*`.
for e in seen:
    if e.event_type.startswith(("plan.", "ui.plan.")):
        print(f"{e.event_type:22s} {e.data}")

plan.init              {'tasks': [{'id': 't1', 'title': 'Generate the dataset', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}, {'id': 't2', 'title': 'Render the chart', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}]}
ui.plan.init           {'component_type': 'plan', 'component_id': 'render-plan', 'data': {'tasks': [{'id': 't1', 'title': 'Generate the dataset', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}, {'id': 't2', 'title': 'Render the chart', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}]}, 'metadata': {}, 'created_at': '2026-05-16T09:01:29.381503Z'}
plan.task.updated      {'task_id': 't1', 'status': 'completed'}
ui.plan.delta          {'component_id': 'render-plan', 'component_type': 'plan', 'op': 'replace', 'path': 'tas

Look at the output above. For each status change, you see:

- a `plan.task.updated` event (legacy shape, used by the existing SSE consumers),
- a `ui.plan.delta` event (typed shape, with an `op`, a `path`, and a `value`).

A frontend renderer wired to `ui.plan.delta` only has to apply that small patch — no full re-render needed.


## 6. The SSE sidecar — turning the bus into a stream

So far we have read events out of memory. In a real web app, the events need to **leave the process** and go to the browser. The standard way to do this is Server-Sent Events (SSE).

`sse_sidecar` does that. It:

- subscribes to the bus,
- yields each event as a properly-formatted SSE string,
- ring-buffers against slow consumers,
- emits heartbeats so connections stay open.

You drop it into a FastAPI `StreamingResponse` and you have a live event feed. Here we just consume two events from it manually so you can see the wire format.


In [6]:
from orqest.observability import sse_sidecar
from orqest.ui import TableComponent, TableComponentData, TableColumn

sidecar = sse_sidecar(wb.event_bus, heartbeat_s=30.0)

# Emit two more components AFTER the sidecar is subscribed.
await emitter.init(chart)

table = TableComponent(
    component_id="editor-table",
    data=TableComponentData(
        columns=[
            TableColumn(key="label", label="Editor"),
            TableColumn(key="value", label="Share"),
        ],
        rows=[{"label": p.label, "value": p.value} for p in dataset.points],
    ),
)
await emitter.init(table)

# Pull two events off the sidecar.
chunks = []
async for chunk in sidecar:
    chunks.append(chunk)
    if len(chunks) >= 2:
        break

for chunk in chunks:
    print(chunk.split(chr(10))[0])      # the "event:" line
    print(f"  ({len(chunk)} bytes of SSE)")

event: ui.chart.init
  (668 bytes of SSE)
event: ui.table.init
  (704 bytes of SSE)


## Recap

Here is the full picture in plain words:

1. A **component** is a typed Pydantic model that describes what should be drawn. There are 17 built-in types, and you can register your own.
2. An **agent** produces structured data. It does not know or care about UI.
3. A `UIEmitter` wraps the data as a component and publishes it on the **event bus** as a `ui.<type>.init` event.
4. For components that change over time (like `ExecutionPlan`), small `ui.<type>.delta` events are sent. The frontend applies them as patches.
5. The `sse_sidecar` turns the bus into a stream that a web frontend can consume directly.

**Why the separation matters.** The agent never imports a UI library. That means the same data agent can drive a web UI, a CLI table, a notebook display, or a test assertion. Swapping the surface costs nothing.

**Two small but useful properties:**

- Bus errors are logged, never raised. A broken UI cannot break the agent.
- `enable_ui_events()` is **dual emission**. New typed events ship without breaking any existing consumer.

You now have everything you need to build agents that drive a live, structured frontend.
